# Data Collection & Preprocessing

In [ ]:
import pandas as pd

# --- 1. COLLECTION ---
df = pd.read_csv('bank_transactions_data_2.csv')

# --- 2. CLEANING ---
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], dayfirst=True)

# --- 3. TRANSFORMATION (Feature Engineering) ---
df['Hour'] = df['TransactionDate'].dt.hour

# --- 4. APPLYING BASIC RULES (Filtering Safe Transactions) ---
is_safe = (df['TransactionAmount'] < 50) & (df['LoginAttempts'] == 1) & (df['TransactionDuration'] > 30)

# We split the data into two buckets
safe_transactions = df[is_safe]
potential_fraud_pool = df[~is_safe] # The '~' means 'NOT safe'

# --- 5. RESULTS ---
print(f"Analysis Complete!")
print(f"Total transactions found: {len(df)}")
print(f"Filtered out as 'Known Safe': {len(safe_transactions)}")
print(f"Remaining for AI analysis: {len(potential_fraud_pool)}")

# Save the 'Potentially Risky' pool to a new file for the next step
potential_fraud_pool.to_csv('pool_for_ai.csv', index=False)

# Anomaly Detection using Unsupervised ML

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- 1. LOAD THE POOL FROM TASK 1 ---
df_pool = pd.read_csv('pool_for_ai.csv')

# --- 2. PREPARE THE DATA ---
le = LabelEncoder()
df_pool['Type_Enc'] = le.fit_transform(df_pool['TransactionType'])
df_pool['Channel_Enc'] = le.fit_transform(df_pool['Channel'])

features = ['TransactionAmount', 'LoginAttempts', 'TransactionDuration', 'AccountBalance', 'Type_Enc', 'Channel_Enc']
X = df_pool[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 3. RUN THE AI MODEL ---
# Finding the most deviant 3% of the pool
model = IsolationForest(contamination=0.03, random_state=42)
df_pool['Anomaly_Label'] = model.fit_predict(X_scaled)
df_pool['Status'] = df_pool['Anomaly_Label'].map({1: 'Normal', -1: 'Suspicious'})

# --- 4. VISUALIZATION ---
plt.figure(figsize=(12, 7))

# We use a scatter plot to show Amount vs Login Attempts
# Red dots = The anomalies found by the AI
sns.scatterplot(data=df_pool, 
                x='TransactionAmount', 
                y='LoginAttempts', 
                hue='Status', 
                palette={'Normal': 'blue', 'Suspicious': 'red'}, 
                alpha=0.7)

plt.title('Task 2: Anomalies Detected in Risky Pool')
plt.xlabel('Transaction Amount ($)')
plt.ylabel('Number of Login Attempts')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- 5. SIMPLE RESULTS ---
suspicious_count = len(df_pool[df_pool['Status'] == 'Suspicious'])
print(f"Total Transactions analyzed: {len(df_pool)}")
print(f"Anomalies detected (Red dots): {suspicious_count}")

# Save for the next step
df_pool[df_pool['Status'] == 'Suspicious'].to_csv('final_anomalies.csv', index=False)

# GenAI-Powered Contextual Dashboard

In [ ]:
%%writefile GEN_AI_FRAUD_DASHBOARD.py

# ===============================
# GEN AI FRAUD DASHBOARD (FIXED)
# ===============================

import streamlit as st
import pandas as pd
import os
from dotenv import load_dotenv
import google.generativeai as genai

st.set_page_config(page_title="Fraud GenAI Dashboard", layout="wide")

# -----------------------
# Load Gemini API
# -----------------------
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    st.error("❌ Gemini API key not found in .env file")
    st.stop()

genai.configure(api_key=api_key)

# -----------------------
# Load anomalies
# -----------------------
@st.cache_data
def load_data():
    return pd.read_csv("final_anomalies.csv")

df_anomalies = load_data()

st.title("🛡️ AI Fraud Detection Dashboard")
st.write("Unsupervised ML + Generative AI Fraud Analysis")

# -----------------------
# Gemini AI reasoning
# -----------------------
def generate_ai_report(row):

    prompt = f"""
    You are a senior fraud analyst in a bank.

    Analyze this suspicious transaction:

    Transaction ID: {row.get('TransactionID','')}
    Amount: {row.get('TransactionAmount','')}
    Account Balance: {row.get('AccountBalance','')}
    Login Attempts: {row.get('LoginAttempts','')}
    Duration: {row.get('TransactionDuration','')}
    Channel: {row.get('Channel','')}
    Type: {row.get('TransactionType','')}

    Explain WHY this looks suspicious in simple business terms.
    Keep answer under 40 words.
    """

    try:
        model = genai.GenerativeModel("models/gemini-2.5-flash")
        response = model.generate_content(prompt)

        # Read Gemini output: 
        if response.candidates:
            return response.candidates[0].content.parts[0].text
        else:
            return "No AI insight generated"

    except Exception as e:
        return f"Gemini Error: {str(e)}"



# -----------------------
# Sidebar filters
# -----------------------
st.sidebar.header("Filters")

min_amount = st.sidebar.slider("Minimum Amount", 0, 5000, 0)

search_id = st.sidebar.text_input("Search Transaction ID")

filtered_df = df_anomalies[df_anomalies["TransactionAmount"] >= min_amount]

if search_id:
    filtered_df = filtered_df[
        filtered_df["TransactionID"].astype(str).str.contains(search_id)
    ]

st.subheader(f"⚠️ Suspicious Transactions Found: {len(filtered_df)}")
st.dataframe(filtered_df, use_container_width=True)

# -----------------------
# AI GENERATE BUTTON
# -----------------------
st.divider()
st.subheader("🧠 Generate AI Fraud Reasoning")

if st.button("Generate AI Insights (Gemini)"):
    
    sample_df = filtered_df.head(20).copy()  # LIMIT to avoid API overload
    
    with st.spinner("Gemini analyzing transactions..."):
        sample_df["AI_Reasoning"] = sample_df.apply(generate_ai_report, axis=1)

    st.success("AI Insights Generated!")

    st.dataframe(sample_df, use_container_width=True)

    # download option
    csv = sample_df.to_csv(index=False).encode("utf-8")
    st.download_button(
        "⬇️ Download AI Report",
        csv,
        "fraud_ai_report.csv",
        "text/csv"
    )


# Automated data pipeline for daily transaction monitoring

In [ ]:
import subprocess
import sys
import time
def my_app():
    App_Path = "GEN_AI_FRAUD_DASHBOARD.py"
    return App_Path
App_Path = my_app()
process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", App_Path])